# Dong Boundary Condition: Steady-State Convergence study computing Kovasznay Flow

First test problem for validating the outflow B.C. by S. Dong. The exact solution for the velocity and pressure fields of the Kovasznay flow is given by:

>$$  \begin{align*} 
u &= 1 - \textrm{e}^{\lambda x} \cos{(2 \pi y)}, \\
v &= \frac{\lambda}{2 \pi} \textrm{e}^{\lambda x} \sin{(2 \pi y)}, \\
p &= \frac{1}{2} (1 - \textrm{e}^{2 \lambda x})
\end{align*}$$

where $\lambda = \frac{1}{2 \nu} - \sqrt{\frac{1}{4 \nu^2} + 4 \pi^2}$ with $\nu = \frac{1}{40}$.

In [22]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Databases loaded: 
Capacity: 0
Count: 0



Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 218
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 105
   at Submission#22.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [23]:
using BoSSS.Solution.NSECommon;

In [24]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfAdditionalServiceCores,0
NumOfAdditionalServiceCoresMPISerial,0
NumOfServiceCoresPerMPIprocess,0
ServerName,DC3


In [25]:
BoSSSshell.WorkflowMgm.Init("KovasznayFlow_ConvStudy", myBatch);

Project name is set to 'KovasznayFlow_ConvStudy'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\KovasznayFlow_ConvStudy'.


In [42]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Grid generation

In [27]:
string BC = IncompressibleBcType.Pressure_Outlet.ToString();
BC

Pressure_Outlet

In [28]:
int[] Resolutions = new int[] { 1, 2, 4, 8, 16, 32, 64, 128, 256 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"KovasznayFlow_{Res}x{2*Res}_{BC}";

    // IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    IGridInfo cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double xend = -0.1;
        double[] xNodes = GenericBlas.Linspace(-0.5, xend, Res + 1);
        double[] yNodes = GenericBlas.Linspace(-0.5, 0.5, (2 * Res) + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes, periodicY:true);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;
            if((X.x + 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.Velocity_Inlet.ToString();
            if((X.x - xend).Abs() <= 1e-8) {
                ret = BC;
            }
            return ret;
        });        
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Grid Edge Tags changed.
An equivalent grid (f7c06097-79ee-4238-93e3-84e05889aea0) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (67353aaf-2cf6-4f76-80c4-fb9d6e2d457f) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (807ba88c-bd31-4d87-a3c6-def8ec460636) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (ff09a55a-5653-41e6-a97d-db12aa649ebd) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (3156a139-8535-44e9-94e6-c4eb99b1704b) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (453726c4-fedb-4389-a07e-c9a40ab5eb43) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (a22b43df-75d0-43f1-9381-7bc31015e0d2) is already present in the data

## Initial Values

In [29]:
Formula KovasznayFlow_u = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velX = 1.0 - (Math.Exp(lambda * X[0]) * Math.Cos(2.0 * Math.PI * X[1]));" +
    "return velX; } "
);

In [30]:
Formula KovasznayFlow_v = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velY = (lambda/(2.0 * Math.PI)) * (Math.Exp(lambda * X[0]) * Math.Sin(2.0 * Math.PI * X[1]));" +
    "return velY; } "
);

In [31]:
Formula KovasznayFlow_p = new Formula(
    "Pres",
    false,
    "double Pres(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double Pres = 0.5 * (1.0 - Math.Exp(2.0 * lambda * X[0]));" +
    "return Pres; } "
);

## Setting up control 

In [32]:
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.DongBoundaryConditionTests;

In [33]:
int[] polyDeg = { 2, 3 };

In [34]:
List<(XNSE_Test_Control ctrl, int numProcs)> Controls = new List<(XNSE_Test_Control, int)>();
Controls.Clear();

In [35]:
bool SolveSystem = true;
bool useManufacturedSol = true;

In [36]:
foreach (int k in polyDeg) {    // loop over poylnomial degrees
foreach (var grd in Grids) {    // loop over grids
//for (int res = 0; res < Resolutions.Length; res++) {
    var C = new XNSE_Test_Control();

    C.SetDGdegree(k);

    // physical parameters
    double rhoSpc = 1.0;
    C.PhysicalParameters.rho_A = rhoSpc;
    double muSpc = 1.0 / 40.0;
    C.PhysicalParameters.mu_A = muSpc;

    C.PhysicalParameters.IncludeConvection = true;

    C.SetGrid(grd);
    int J = grd.NumberOfCells * ((((k + 1)*(k + 1)) + (k + 1)) / 2);
    int nP = (J / 250000) + 1;    

    // initial conditions
    if (!SolveSystem) {
        C.SkipSolveAndEvaluateResidual = true;

        C.AddInitialValue("VelocityX#A", KovasznayFlow_u);
        C.AddInitialValue("VelocityY#A", KovasznayFlow_v);
        C.AddInitialValue("Pressure#A", KovasznayFlow_p);
    }

    // boundary conditions
    C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", KovasznayFlow_u);
    C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityY#A", KovasznayFlow_v);

    C.UseManufacturedComps = useManufacturedSol;

    //C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    //C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.Timestepper_LevelSetHandling = LevelSetHandling.None;
    C.Option_LevelSetEvolution = LevelSetEvolution.None;

    // C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    // C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    // C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    // C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
    // C.dtFixed = 2.5e-5;
    // C.NoOfTimesteps = 4; 

    C.SessionName = useManufacturedSol ? $"KovasznayFlow_ConvStudy_{BC}_ManSol_p{k}_J{J}" : $"KovasznayFlow_ConvStudy_{BC}_p{k}_J{J}";
    
    Controls.Add((C, nP));
}
}

In [37]:
Controls.Count

18

In [38]:
//Controls.Select(ctrl => ctrl.SessionName)
Controls.Pick(0).ctrl.SessionName

KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p2_J12

In [39]:
Controls.Select(C => C.numProcs)

[ 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 2, 6 ]

In [40]:
myBatch.Name

FDY-WindowsHPC

In [41]:
//Controls.Pick(0).Run();

In [43]:
foreach ((XNSE_Test_Control ctrl, int numProcs) in Controls) {
    var oneJob              = ctrl.CreateJob();
    oneJob.RetryCount = 1;
    
    oneJob.NumberOfMPIProcs = numProcs;

    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

   
    if (myBatch.Name == "Lb2-specialPrj") {
        //oneJob.UseComputeNodesExclusive = true;
        ((SlurmClient)myBatch).ExecutionTime = "24:00:00";
    }

    Console.WriteLine($"Activate job: {ctrl.SessionName} with {numProcs} MPI cores");
    oneJob.Activate(myBatch); 
}

Activate job: KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p2_J12 with 1 MPI cores
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p2_J12 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\KovasznayFlow_ConvStudy-DongBoundaryConditionTests2025Apr16_140008.290220
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Activate job: KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p2_J48 with 1 MPI cores
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p2_J48 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\KovasznayFlow_ConvStudy-DongBoundaryConditionTests2025Apr16_140022.990734
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Activate job: KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p2_J192 with 1 MPI cores
Deployments so far (0): ;
Success: 0
job submit count: 0


...

Deploying job KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p3_J81920 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\KovasznayFlow_ConvStudy-DongBoundaryConditionTests2025Apr16_140344.847605
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Activate job: KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p3_J327680 with 2 MPI cores
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p3_J327680 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\KovasznayFlow_ConvStudy-DongBoundaryConditionTests2025Apr16_140359.498897
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Activate job: KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p3_J1310720 with 6 MPI cores
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p3_J1310720 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\KovasznayFlow_ConvStudy-DongBoundaryConditionTests2025Apr16_140414.128170
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
